In [ ]:
import torch
from torch.utils.data import DataLoader, random_split
from utilities2 import (
    KolmogorovDataset,
    FNOGenerator,
    FNODiscriminatorStat,
    FNODiscriminatorPhys,
    NavierStokesResiduo,
    WGAFNOGPTrainer,
    Rollout,
)

In [ ]:
# ── Config ─────────────────────────────────────────────────
DEVICE    = "cuda" if torch.cuda.is_available() else "cpu"
DATA_PATH = "../datasets/Snapshots/snapshots_64x64_use.npy"
H, W      = 64, 64
SEQ_LEN   = 10
EPOCHS    = 30

BATCH    = 8
MODES    = 12
HIDDEN   = 32
N_LAYERS = 4
N_CRITIC = 2
Z_DIM    = 8

DT        = 0.05
LAMBDA_GP = 10.0

In [ ]:
# ── Dataset ────────────────────────────────────────────────
dataset   = KolmogorovDataset(DATA_PATH, seq_len=SEQ_LEN)
train_len = int(0.8 * len(dataset))
val_len   = len(dataset) - train_len
train_ds, val_ds = random_split(dataset, [train_len, val_len])

train_loader = DataLoader(
    train_ds, batch_size=BATCH, shuffle=True,
    num_workers=4, pin_memory=True, persistent_workers=True,
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH, shuffle=False, num_workers=4,
)


In [ ]:
# ── Modelos ────────────────────────────────────────────────
G = FNOGenerator(
    hidden_ch=HIDDEN, modes1=MODES, modes2=MODES,
    n_layers=N_LAYERS, z_dim=Z_DIM,
).to(DEVICE)

D_stat = FNODiscriminatorStat(
    seq_len=SEQ_LEN + 1, hidden_ch=HIDDEN,
    modes1=MODES, modes2=MODES, n_layers=N_LAYERS,
).to(DEVICE)

D_phys = FNODiscriminatorPhys(
    seq_len=SEQ_LEN + 1, hidden_ch=HIDDEN,
    modes1=MODES, modes2=MODES, n_layers=N_LAYERS,
).to(DEVICE)

ns_residuo = NavierStokesResiduo(
    H, W, dt=DT, modes1=MODES, modes2=MODES, device=DEVICE,
).to(DEVICE)

In [1]:
# ── Trainer ────────────────────────────────────────────────
trainer = WGAFNOGPTrainer(
    generator          = G,
    d_stat             = D_stat,
    d_phys             = D_phys,
    ns_residuo         = ns_residuo,
    device             = DEVICE,
    lr_G               = 1e-4,
    lr_D               = 1e-4,
    n_critic           = N_CRITIC,
    lambda_gp          = LAMBDA_GP,
    use_scheduler      = True,
    scheduler_patience = 5,
    scheduler_factor   = 0.5,
    log_dir            = "training_logs",
    vis_freq           = 1,
)

# ── Entrenamiento ──────────────────────────────────────────
history = trainer.fit(train_loader, val_loader, n_epochs=EPOCHS, log_every=1)
history

2026-04-03 04:50:51,734 | INFO | Dataset: 1152 trayectorias × 90 ventanas = 103,680 muestras | seq_len=10 | H×W=64×64
Epoch    1/30:   0%|                                  | 0/10368 [00:00<?, ?it/s]/home/joseluis/miniconda3/envs/cfd-project-personal/lib/python3.10/site-packages/torch/autograd/graph.py:824: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at /pytorch/aten/src/ATen/cuda/CublasHandlePool.cpp:181.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
2026-04-03 05:50:43,170 | INFO | ✅ Nuevo mejor — época 1, val MSE=0.527931      
/media/joseluis/HDD/Proyecto CFD/tasks/utilities2.py:803: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()
2026-04-03 05:50:45,217 | INFO |   Campos → training_logs/fields_epoch0001.png
2026-04-03 05:50:45,874 | INFO |   Es

{'loss_D_stat': [-5.605563350884175,
  -33.254194504684875,
  -45.09122699443941,
  -58.73399265110493,
  -62.39462885922856,
  -63.71976721121205,
  -66.09153136335037,
  -68.6889726922468,
  -69.9875615708254,
  -70.84226298608162,
  -69.39928368727367,
  -69.2639280292723,
  -70.72161039323719,
  -71.11209569761047,
  -71.85404974001425,
  -72.3064449114932,
  -73.38286953502231,
  -73.86740037688503,
  -74.43182120610166,
  -74.23816709054842,
  -73.76658273791826,
  -74.00610407524638,
  -74.3497888788029,
  -74.36586282319493,
  -74.63313474809682,
  -74.36445249341152,
  -74.14958969696805,
  -74.3541050752004,
  -73.98032126658492,
  -73.88507478711782],
 'loss_D_phys': [-11.423422777162934,
  -88.05444202268565,
  -190.84036563060903,
  -293.79291925827664,
  -382.50790019167795,
  -435.4345681976389,
  -473.0383646024598,
  -499.0125912891494,
  -530.2911784935881,
  -558.5607781630976,
  -557.1632406005153,
  -587.6085192274164,
  -624.5169125707062,
  -648.7552974179939,
  